In [1]:


import kagglehub
import os

path = kagglehub.dataset_download("shaunthesheep/microsoft-catsvsdogs-dataset")
print("Dataset path:", path)

dataset_path = os.path.join(path, "PetImages")
print(os.listdir(dataset_path))

Using Colab cache for faster access to the 'microsoft-catsvsdogs-dataset' dataset.
Dataset path: /kaggle/input/microsoft-catsvsdogs-dataset
['Dog', 'Cat']


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

In [3]:

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [4]:
import tensorflow as tf
import os

dataset_path = os.path.join(path, "PetImages")

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224,224),
    batch_size=16,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224,224),
    batch_size=16,
    label_mode="binary"
)

train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
val_ds = val_ds.apply(tf.data.experimental.ignore_errors())

Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


In [5]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1)
])

model = models.Sequential([
    layers.Input(shape=(224,224,3)),
    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32,(3,3),padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64,(3,3),padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128,(3,3),padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(256,(3,3),padding='same',activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128,activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(64,activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(1,activation='sigmoid')
])

In [6]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [7]:
# Early stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3
)

In [8]:

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
   1243/Unknown 96s 70ms/step - accuracy: 0.5690 - loss: 0.7030

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1243/1243 ━━━━━━━━━━━━━━━━━━━━ 112s 83ms/step - accuracy: 0.5690 - loss: 0.7030 - val_accuracy: 0.6793 - val_loss: 0.5997
Epoch 2/20
1243/1243 ━━━━━━━━━━━━━━━━━━━━ 100s 80ms/step - accuracy: 0.6414 - loss: 0.6287 - val_accuracy: 0.6959 - val_loss: 0.5873
Epoch 3/20
1243/1243 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.6712 - loss: 0.6066 - val_accuracy: 0.7172 - val_loss: 0.5551
Epoch 4/20
1243/1243 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.6830 - loss: 0.5906 - val_accuracy: 0.7073 - val_loss: 0.5767
Epoch 5/20
1243/1243 ━━━━━━━━━━━━━━━━━━━━ 125s 80ms/step - accuracy: 0.7069 - loss: 0.5673 - val_accuracy: 0.7351 - val_loss: 0.5313
Epoch 6/20
1243/1243 ━━━━━━━━━━━━━━━━━━━━ 99s 80ms/step - accuracy: 0.7145 - loss: 0.5651 - val_accuracy: 0.7530 - val_loss: 0.5082
Epoch 7/20
1243/1243 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.7193 - loss: 0.5488 - val_accuracy: 0.7587 - val_loss: 0.4964
Epoch 8/20
1243/1243 ━━━━━━━━━━━━━━━━━━━━ 99s 79ms/step - accuracy: 0.7342 - loss: 0

In [9]:
loss, accuracy = model.evaluate(val_ds)

print("Accuracy:", accuracy)
print("Loss:", loss)

311/311 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - accuracy: 0.8587 - loss: 0.3216
Accuracy: 0.8611111044883728
Loss: 0.32008397579193115
